# 🏭 IA Neurosimbólica — Control de Calidad en Manufactura de Cable

**Caso de uso:** Una planta fabricante de cable eléctrico necesita detectar defectos en sus líneas de producción y tomar decisiones de disposición (aprobar / reparar / rechazar). El sistema no solo debe predecir si hay un defecto, sino **explicar por qué** y aplicar las **reglas de la norma NOM/IEC** correspondientes.

---
### ¿Qué hace este notebook?

| Capa | Componente | Qué simula |
|------|-----------|------------|
| **Percepción neural** | Random Forest sobre mediciones de producción | El modelo de ML que aprende a detectar anomalías |
| **Integración** | Conversión de probabilidades a hechos simbólicos | El puente entre el score y la lógica |
| **Razonamiento simbólico** | Motor de reglas de norma IEC/NOM | Las reglas de calidad explícitas y auditables |

> **Nota:** Los datos son sintéticos pero basados en parámetros reales de fabricación de cable (resistencia eléctrica, espesor de aislante, excentricidad, temperatura de extrusión).

## 📦 Celda 1 — Instalación e importación de librerías

Instalamos e importamos las librerías necesarias:
- `numpy` y `pandas`: para manipular los datos de producción.
- `scikit-learn`: para entrenar el modelo neuronal (Random Forest).
- `matplotlib` y `seaborn`: para visualizar resultados.
- `warnings`: para suprimir mensajes de deprecación que no son relevantes para el análisis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Librerías cargadas correctamente')

## 🏗️ Celda 2 — Generación del dataset de producción de cable

Simulamos 2,000 rollos de cable producidos en planta con las siguientes variables de proceso:

| Variable | Unidad | Rango normal | Descripción |
|----------|--------|-------------|-------------|
| `resistencia_ohm_km` | Ω/km | 9.5 – 10.5 | Resistencia eléctrica del conductor de cobre |
| `espesor_aislante_mm` | mm | 0.8 – 1.2 | Grosor del recubrimiento de PVC |
| `excentricidad_pct` | % | < 10% | Desplazamiento del conductor respecto al centro |
| `temp_extrusion_c` | °C | 170 – 185 | Temperatura de la extrusora durante producción |
| `velocidad_linea_mpm` | m/min | 40 – 60 | Velocidad de la línea de producción |
| `tension_halado_n` | N | 80 – 120 | Tensión de halado del conductor |

Los defectos se etiquetan como `1` cuando alguno de estos parámetros sale de especificación de forma significativa.

In [ ]:
N = 2000

# --- Variables de proceso (distribuciones normales alrededor de especificación) ---
resistencia    = np.random.normal(10.0, 0.4, N)       # Ω/km, nominal 10.0
espesor        = np.random.normal(1.0,  0.12, N)       # mm, nominal 1.0
excentricidad  = np.abs(np.random.normal(5.0, 3.5, N)) # %, siempre positivo
temp_extrusion = np.random.normal(178, 6, N)           # °C
velocidad      = np.random.normal(50, 8, N)            # m/min
tension_halado = np.random.normal(100, 15, N)          # N

# --- Etiqueta de defecto: combinación de condiciones fuera de especificación ---
defecto = (
    (resistencia < 9.2) | (resistencia > 10.8) |          # fuera de tolerancia eléctrica
    (espesor < 0.75)    | (espesor > 1.25)    |           # aislante demasiado delgado o grueso
    (excentricidad > 12)                       |           # descentrado excesivo
    (temp_extrusion < 168) | (temp_extrusion > 192) |     # temperatura crítica
    ((velocidad > 62) & (espesor < 0.85))                 # velocidad alta + aislante delgado = riesgo
).astype(int)

df = pd.DataFrame({
    'resistencia_ohm_km':   np.round(resistencia, 3),
    'espesor_aislante_mm':  np.round(espesor, 3),
    'excentricidad_pct':    np.round(excentricidad, 2),
    'temp_extrusion_c':     np.round(temp_extrusion, 1),
    'velocidad_linea_mpm':  np.round(velocidad, 1),
    'tension_halado_n':     np.round(tension_halado, 1),
    'defecto':              defecto
})

print(f'Dataset generado: {len(df)} rollos de cable')
print(f'  → Con defecto:     {defecto.sum():>4} ({defecto.mean()*100:.1f}%)')
print(f'  → Sin defecto:     {(1-defecto).sum():>4} ({(1-defecto).mean()*100:.1f}%)')
df.head(8)

## 📊 Celda 3 — Análisis exploratorio: distribución de parámetros

Visualizamos cómo se distribuyen las variables de proceso diferenciando rollos buenos (azul) de defectuosos (rojo). Esto nos permite **validar visualmente** que los datos tienen sentido antes de entrenar el modelo.

In [ ]:
features = ['resistencia_ohm_km', 'espesor_aislante_mm', 'excentricidad_pct',
            'temp_extrusion_c', 'velocidad_linea_mpm', 'tension_halado_n']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Distribución de parámetros de producción\nAzul = sin defecto | Rojo = con defecto',
             fontsize=13, y=1.01)

colores = {0: '#3B82F6', 1: '#EF4444'}
etiquetas = {0: 'Sin defecto', 1: 'Con defecto'}

for ax, feat in zip(axes.flatten(), features):
    for clase, color in colores.items():
        subset = df[df['defecto'] == clase][feat]
        ax.hist(subset, bins=30, alpha=0.55, color=color,
                label=etiquetas[clase], density=True)
    ax.set_title(feat.replace('_', ' '), fontsize=10)
    ax.set_ylabel('Densidad', fontsize=8)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print('Observación: los rollos defectuosos tienen colas más pronunciadas en resistencia, espesor y excentricidad.')

## 🧠 Celda 4 — CAPA NEURAL: Entrenamiento del modelo de detección

Aquí entra la **primera capa de la IA Neurosimbólica**: el componente de percepción neural.

Usamos un **Random Forest** que aprende, a partir de los datos históricos de producción, a identificar qué combinaciones de parámetros son indicativas de defecto. El modelo **no conoce las reglas de la norma** — solo aprende patrones estadísticos.

Al final, el modelo nos devuelve para cada rollo:
- `prediccion_defecto`: 0 o 1
- `prob_defecto`: probabilidad continua entre 0 y 1 — esta será la entrada a la capa de integración.

In [ ]:
X = df[features]
y = df['defecto']

# División 80/20 entrenamiento / prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalización (recomendada aunque RF no la requiere, sirve para la capa de integración)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Entrenamiento del modelo neuronal
modelo_rf = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, class_weight='balanced')
modelo_rf.fit(X_train_sc, y_train)

# Predicciones y probabilidades
y_pred      = modelo_rf.predict(X_test_sc)
y_prob      = modelo_rf.predict_proba(X_test_sc)[:, 1]  # probabilidad de defecto

print('=== CAPA NEURAL: Reporte de desempeño ===')
print(classification_report(y_test, y_pred, target_names=['Sin defecto', 'Con defecto']))

# Matriz de confusión
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: OK', 'Pred: Defecto'],
            yticklabels=['Real: OK', 'Real: Defecto'])
ax.set_title('Matriz de confusión — Capa Neural', fontsize=11)
plt.tight_layout()
plt.show()

## 🔗 Celda 5 — CAPA DE INTEGRACIÓN: Conversión de scores a hechos simbólicos

La capa de integración **traduce el lenguaje de las probabilidades** (salida de la red neuronal) al **lenguaje de la lógica** (entrada del motor de reglas).

Para cada rollo, el sistema construye un **diccionario de hechos** con:
1. El nivel de riesgo detectado por el modelo (`bajo`, `medio`, `alto`, `critico`)
2. Qué parámetros específicos están fuera de especificación (verificación directa sobre los valores medidos)
3. La combinación de factores que determinará la decisión final

Esta es la clave de la IA Neurosimbólica: **el modelo aprende, pero las reglas deciden**.

In [ ]:
def extraer_hechos(fila, prob_defecto):
    """
    Capa de integración: convierte mediciones + probabilidad en hechos simbólicos.
    Retorna un diccionario con el estado de cada parámetro y el nivel de riesgo.
    """
    hechos = {}

    # Nivel de riesgo según la probabilidad del modelo neuronal
    if   prob_defecto < 0.25: hechos['riesgo_neuronal'] = 'bajo'
    elif prob_defecto < 0.50: hechos['riesgo_neuronal'] = 'medio'
    elif prob_defecto < 0.75: hechos['riesgo_neuronal'] = 'alto'
    else:                     hechos['riesgo_neuronal'] = 'critico'

    # Hechos simbólicos: cada parámetro se evalúa contra su especificación
    hechos['resistencia_fuera'] = not (9.5 <= fila['resistencia_ohm_km'] <= 10.5)
    hechos['aislante_delgado']  = fila['espesor_aislante_mm'] < 0.80
    hechos['aislante_grueso']   = fila['espesor_aislante_mm'] > 1.20
    hechos['excentricidad_alta']= fila['excentricidad_pct'] > 10.0
    hechos['temp_baja']         = fila['temp_extrusion_c'] < 170
    hechos['temp_alta']         = fila['temp_extrusion_c'] > 188
    hechos['velocidad_alta']    = fila['velocidad_linea_mpm'] > 60

    # Conteo de parámetros fuera de especificación
    hechos['n_params_fuera'] = sum([
        hechos['resistencia_fuera'], hechos['aislante_delgado'], hechos['aislante_grueso'],
        hechos['excentricidad_alta'], hechos['temp_baja'], hechos['temp_alta']
    ])

    return hechos

# Demostración con un rollo de ejemplo
idx_ejemplo = 5
fila_demo = X_test.iloc[idx_ejemplo]
prob_demo  = y_prob[idx_ejemplo]
hechos_demo = extraer_hechos(fila_demo, prob_demo)

print(f'=== CAPA DE INTEGRACIÓN: Hechos extraídos para rollo de ejemplo ===')
print(f'Probabilidad de defecto (modelo neural): {prob_demo:.3f}')
print()
for hecho, valor in hechos_demo.items():
    icono = '⚠️' if valor is True else ('✅' if valor is False else '📊')
    print(f'  {icono}  {hecho}: {valor}')

## 📋 Celda 6 — CAPA SIMBÓLICA: Motor de reglas de norma IEC/NOM

Aquí entra la **capa de razonamiento simbólico**. El motor de reglas codifica el conocimiento experto de la norma **IEC 60227** (cables de PVC para tensiones hasta 450/750V) y las políticas internas de calidad.

Las reglas son explícitas, auditables y pueden ser modificadas por el equipo de ingeniería sin reentrenar el modelo. Producen:
- **Decisión:** `APROBAR`, `CUARENTENA`, `REPROCESAR` o `RECHAZAR`
- **Justificación:** texto legible que explica exactamente por qué se tomó la decisión
- **Referencia normativa:** la cláusula de la norma que aplica

In [ ]:
def motor_reglas_cable(hechos, fila):
    """
    Motor de razonamiento simbólico basado en norma IEC 60227 / NOM-001-SEDE.
    Retorna: (decision, justificacion, norma_referencia)
    """
    r = fila['resistencia_ohm_km']
    e = fila['espesor_aislante_mm']
    exc = fila['excentricidad_pct']
    temp = fila['temp_extrusion_c']

    # ─── REGLA 1: Defecto crítico de seguridad eléctrica ───────────────────────
    # Si el aislante está por debajo del mínimo de seguridad, es rechazo inmediato
    if hechos['aislante_delgado'] and e < 0.70:
        return (
            'RECHAZAR',
            f'Espesor de aislante {e:.3f} mm es menor al mínimo absoluto de 0.70 mm. '
            f'Riesgo de falla eléctrica en campo.',
            'IEC 60227-2 Cláusula 3.2 — Espesor nominal mínimo de aislamiento'
        )

    # ─── REGLA 2: Resistencia fuera de especificación con riesgo alto ──────────
    if hechos['resistencia_fuera'] and hechos['riesgo_neuronal'] in ('alto', 'critico'):
        return (
            'RECHAZAR',
            f'Resistencia {r:.3f} Ω/km fuera de rango 9.5–10.5 Ω/km y el modelo '
            f'neuronal detecta riesgo {hechos["riesgo_neuronal"].upper()}. '
            f'Cable no cumple especificación eléctrica.',
            'IEC 60228 Clase 1 — Resistencia máxima de conductores a 20°C'
        )

    # ─── REGLA 3: Múltiples parámetros fuera de spec ──────────────────────────
    if hechos['n_params_fuera'] >= 2:
        return (
            'RECHAZAR',
            f'{hechos["n_params_fuera"]} parámetros simultáneamente fuera de especificación. '
            f'La combinación de fallas indica problema sistémico de proceso.',
            'Procedimiento interno QC-MFG-003 Rev.4 — Disposición por fallas múltiples'
        )

    # ─── REGLA 4: Excentricidad alta — reproceso posible ─────────────────────
    if hechos['excentricidad_alta'] and exc > 10:
        return (
            'REPROCESAR',
            f'Excentricidad {exc:.1f}% supera el límite de 10%. Se puede corregir '
            f'ajustando la cabeza de extrusión. Aislante intacto.',
            'IEC 60811-1-1 — Medición de excentricidad de aislantes'
        )

    # ─── REGLA 5: Temperatura anormal + riesgo medio ──────────────────────────
    if (hechos['temp_baja'] or hechos['temp_alta']) and hechos['riesgo_neuronal'] == 'medio':
        return (
            'CUARENTENA',
            f'Temperatura de extrusión {temp:.1f}°C fuera de rango 170–188°C. '
            f'El modelo neuronal detecta riesgo MEDIO. Requiere inspección adicional '
            f'de propiedades mecánicas del aislante.',
            'IEC 60811-1-2 — Ensayos de envejecimiento del aislamiento'
        )

    # ─── REGLA 6: Riesgo neuronal bajo con aislante dentro de spec ────────────
    if hechos['riesgo_neuronal'] == 'bajo' and not hechos['resistencia_fuera']:
        return (
            'APROBAR',
            f'Todos los parámetros críticos dentro de especificación. '
            f'El modelo neuronal asigna riesgo BAJO (prob={hechos.get("prob", "N/A")}). '
            f'Rollo aprobado para despacho.',
            'NOM-001-SEDE-2012 — Criterio de aceptación de lotes'
        )

    # ─── REGLA DEFAULT: Cuarentena para revisión manual ──────────────────────
    return (
        'CUARENTENA',
        f'El modelo neuronal detecta riesgo {hechos["riesgo_neuronal"].upper()} pero '
        f'las reglas no encuentran falla específica. Se requiere inspección visual manual.',
        'Procedimiento interno QC-MFG-001 Rev.7 — Disposición por criterio de operador'
    )

# Prueba con el ejemplo anterior
decision, justif, norma = motor_reglas_cable(hechos_demo, fila_demo)

iconos = {'APROBAR': '✅', 'CUARENTENA': '🟡', 'REPROCESAR': '🔧', 'RECHAZAR': '🔴'}
print(f'{iconos[decision]} DECISIÓN: {decision}')
print(f'📝 Justificación: {justif}')
print(f'📌 Norma: {norma}')

## 🔄 Celda 7 — Pipeline completo: procesar todos los rollos del turno

Combinamos las tres capas en un pipeline que procesa todos los rollos del conjunto de prueba. Este sería el flujo en tiempo real durante un turno de producción:

1. Los sensores de la línea envían mediciones → dataset
2. El modelo neuronal calcula la probabilidad de defecto → capa neural
3. Las mediciones + probabilidad se convierten en hechos → capa integración
4. El motor de reglas emite una decisión explicada → capa simbólica
5. El operador ve la decisión con justificación en pantalla

In [ ]:
resultados = []

for i, (idx, fila) in enumerate(X_test.iterrows()):
    prob = y_prob[i]
    hechos = extraer_hechos(fila, prob)
    hechos['prob'] = f'{prob:.3f}'
    decision, justif, norma = motor_reglas_cable(hechos, fila)

    resultados.append({
        'rollo_id':             f'ROLLO-{idx:04d}',
        'prob_defecto_neural':  round(prob, 3),
        'riesgo_neuronal':      hechos['riesgo_neuronal'],
        'decision':             decision,
        'justificacion':        justif,
        'norma_referencia':     norma,
        'defecto_real':         y_test.iloc[i]
    })

df_resultados = pd.DataFrame(resultados)

print('=== RESUMEN DEL TURNO ===')
print(df_resultados['decision'].value_counts().to_string())
print(f'\nTotal rollos procesados: {len(df_resultados)}')
print()
print('Muestra de resultados con justificación:')
df_resultados[['rollo_id', 'prob_defecto_neural', 'riesgo_neuronal', 'decision']].head(10)

## 📊 Celda 8 — Visualización: dashboard de disposición del turno

Generamos el dashboard de resultados que vería el supervisor de turno en planta. Muestra la distribución de decisiones y la relación entre el riesgo neuronal y la disposición final.

**Lo clave de la IA Neurosimbólica aquí:** un rollo puede tener riesgo neuronal ALTO pero ser marcado como REPROCESAR (no RECHAZAR) si la única falla es de excentricidad — porque la regla de negocio lo permite. El modelo solo no lo sabría.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Dashboard IA Neurosimbólica — Control de Calidad Cable', fontsize=13, fontweight='bold')

colores_decision = {
    'APROBAR':    '#22C55E',
    'CUARENTENA': '#F59E0B',
    'REPROCESAR': '#3B82F6',
    'RECHAZAR':   '#EF4444'
}

# Gráfico 1: Distribución de decisiones
conteo = df_resultados['decision'].value_counts()
colores_bar = [colores_decision[d] for d in conteo.index]
bars = axes[0].bar(conteo.index, conteo.values, color=colores_bar, edgecolor='white', linewidth=0.8)
axes[0].set_title('Disposición del turno', fontsize=11)
axes[0].set_ylabel('Número de rollos')
for bar, val in zip(bars, conteo.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

# Gráfico 2: Riesgo neuronal vs decisión simbólica (mapa de calor)
orden_riesgo = ['bajo', 'medio', 'alto', 'critico']
orden_decision = ['APROBAR', 'CUARENTENA', 'REPROCESAR', 'RECHAZAR']
tabla_cruzada = pd.crosstab(
    df_resultados['riesgo_neuronal'],
    df_resultados['decision']
).reindex(index=orden_riesgo, columns=orden_decision, fill_value=0)

sns.heatmap(tabla_cruzada, ax=axes[1], annot=True, fmt='d', cmap='YlOrRd',
            cbar_kws={'label': 'Num. rollos'})
axes[1].set_title('Riesgo neural → Decisión simbólica', fontsize=11)
axes[1].set_xlabel('Decisión simbólica')
axes[1].set_ylabel('Riesgo neuronal')

# Gráfico 3: Distribución de probabilidades por decisión
for decision, color in colores_decision.items():
    subset = df_resultados[df_resultados['decision'] == decision]['prob_defecto_neural']
    if len(subset) > 0:
        axes[2].hist(subset, bins=15, alpha=0.6, color=color,
                     label=f'{decision} (n={len(subset)})', density=True)
axes[2].set_title('Distribución de riesgo neuronal\npor tipo de decisión', fontsize=11)
axes[2].set_xlabel('Probabilidad de defecto')
axes[2].set_ylabel('Densidad')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 🔍 Celda 9 — Trazabilidad: reporte de un rollo específico

Una de las grandes ventajas de la IA Neurosimbólica es la **trazabilidad completa**. Si un cliente reclama, o si hay una auditoría, el sistema puede imprimir exactamente **qué midió, qué detectó el modelo, qué regla aplicó y qué norma lo sustenta**.

Esto no es posible con una red neuronal sola — el "por qué" quedaría perdido dentro de los pesos del modelo.

In [ ]:
def reporte_trazabilidad(rollo_id, df_resultados, X_test, y_prob, y_test):
    """Imprime el reporte de trazabilidad completo para un rollo específico."""

    fila_res = df_resultados[df_resultados['rollo_id'] == rollo_id].iloc[0]
    idx_num  = int(rollo_id.split('-')[1])

    # Recuperar la fila original
    try:
        fila_orig = X_test.loc[idx_num]
        prob      = y_prob[X_test.index.get_loc(idx_num)]
        real      = y_test.loc[idx_num]
    except Exception:
        print(f'Rollo {rollo_id} no encontrado en el conjunto de prueba.')
        return

    iconos = {'APROBAR': '✅', 'CUARENTENA': '🟡', 'REPROCESAR': '🔧', 'RECHAZAR': '🔴'}
    sep = '─' * 55

    print(f'\n{sep}')
    print(f'  REPORTE DE TRAZABILIDAD — {rollo_id}')
    print(f'{sep}')
    print(f'  Fecha/Turno: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}')
    print()
    print('  [1] MEDICIONES DE PROCESO')
    especificaciones = {
        'resistencia_ohm_km': (9.5, 10.5, 'Ω/km'),
        'espesor_aislante_mm': (0.80, 1.20, 'mm'),
        'excentricidad_pct': (0, 10.0, '%'),
        'temp_extrusion_c': (170, 188, '°C'),
        'velocidad_linea_mpm': (30, 65, 'm/min'),
        'tension_halado_n': (60, 140, 'N'),
    }
    for col, (lo, hi, unit) in especificaciones.items():
        val = fila_orig[col]
        estado = '✅ OK' if lo <= val <= hi else '⚠️  FUERA DE SPEC'
        print(f'    {col:<26} {val:>8.2f} {unit:<7} [{lo}–{hi}]  {estado}')

    print()
    print(f'  [2] CAPA NEURAL')
    print(f'    Probabilidad de defecto : {prob:.4f}')
    print(f'    Nivel de riesgo         : {fila_res["riesgo_neuronal"].upper()}')
    print(f'    Etiqueta real           : {"DEFECTO" if real == 1 else "OK"}')

    print()
    print(f'  [3] CAPA SIMBÓLICA — DECISIÓN FINAL')
    print(f'    Decisión  : {iconos[fila_res["decision"]]} {fila_res["decision"]}')
    print(f'    Motivo    : {fila_res["justificacion"]}')
    print(f'    Normativa : {fila_res["norma_referencia"]}')
    print(f'{sep}\n')

# Reportes de ejemplo: uno aprobado, uno rechazado
aprobados  = df_resultados[df_resultados['decision'] == 'APROBAR']['rollo_id']
rechazados = df_resultados[df_resultados['decision'] == 'RECHAZAR']['rollo_id']

if len(aprobados) > 0:
    reporte_trazabilidad(aprobados.iloc[0], df_resultados, X_test, y_prob, y_test)
if len(rechazados) > 0:
    reporte_trazabilidad(rechazados.iloc[0], df_resultados, X_test, y_prob, y_test)

## 🏆 Celda 10 — Conclusiones: ¿Qué aporta la IA Neurosimbólica vs. solo ML?

Comparamos qué habría pasado si solo hubiéramos usado el modelo neuronal (Random Forest) contra el sistema completo.

In [ ]:
# ¿Cuántos rollos clasificó el modelo neural vs la capa simbólica de forma diferente?
df_resultados['pred_neural_sola'] = (df_resultados['prob_defecto_neural'] >= 0.5).map(
    {True: 'RECHAZAR (neural)', False: 'APROBAR (neural)'}
)

# Rollos donde la capa simbólica matizó la decisión
matizados = df_resultados[
    ((df_resultados['prob_defecto_neural'] >= 0.5) & (df_resultados['decision'] != 'RECHAZAR')) |
    ((df_resultados['prob_defecto_neural'] < 0.5)  & (df_resultados['decision'] == 'RECHAZAR'))
]

print('=== VALOR AÑADIDO DE LA CAPA SIMBÓLICA ===')
print(f'Total rollos procesados          : {len(df_resultados)}')
print(f'Decisiones matizadas por reglas  : {len(matizados)} ({len(matizados)/len(df_resultados)*100:.1f}%)')
print()
print('Ejemplos de matizaciones simbólicas:')
print(matizados[['rollo_id', 'prob_defecto_neural', 'riesgo_neuronal', 'decision', 'norma_referencia']].head(5).to_string(index=False))

print('\n=== RESUMEN CONCEPTUAL ===')
print("""
  ┌─────────────────────┬───────────────────────────────────────────────────┐
  │ Solo red neuronal   │ Solo IA simbólica                                 │
  ├─────────────────────┼───────────────────────────────────────────────────┤
  │ • Detecta patrones  │ • Aplica reglas IEC/NOM explícitas                │
  │ • No explica el     │ • No aprende de datos nuevos                      │
  │   porqué            │ • Necesita ingenieros para escribir las reglas    │
  │ • No conoce normas  │                                                   │
  ├─────────────────────┴───────────────────────────────────────────────────┤
  │ IA NEUROSIMBÓLICA: aprende Y razona Y explica Y cumple normas           │
  └─────────────────────────────────────────────────────────────────────────┘
""")